# EDA + Modelo Predictivo de Ganador en Combates Pokémon

## Objetivo
Construir un pipeline reproducible para:
1. Analizar calidad y estructura de los datos.
2. Crear features de enfrentamiento entre dos Pokémon.
3. Entrenar modelos para predecir el ganador.
4. Evaluar desempeño e interpretar variables relevantes.

## Criterio de éxito
- Modelo con desempeño superior a un baseline simple.
- Proceso sin fuga de información (data leakage).
- Resultados interpretables para toma de decisiones.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

SEED = 42
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_DIR = Path('../data')
POKEMON_PATH = DATA_DIR / 'pokemon.csv'
COMBATS_PATH = DATA_DIR / 'combats.csv'

print('POKEMON_PATH:', POKEMON_PATH.resolve())
print('COMBATS_PATH:', COMBATS_PATH.resolve())

## 1) Carga de datos
Se cargan:
- `pokemon.csv`: atributos base de cada Pokémon.
- `combats.csv`: pares de enfrentamiento y ganador.

Validaremos dimensiones y una muestra inicial.

In [ ]:
pokemon = pd.read_csv(POKEMON_PATH)
combats = pd.read_csv(COMBATS_PATH)

print('pokemon shape:', pokemon.shape)
print('combats shape:', combats.shape)

display(pokemon.head())
display(combats.head())

## 2) Calidad de datos y consistencia
Revisamos:
- Tipos de datos y nulos.
- Duplicados.
- Consistencia de IDs de combates contra catálogo de Pokémon.

Esto evita errores silenciosos en el modelado.

In [ ]:
print('--- pokemon info ---')
pokemon.info()

print('\n--- combats info ---')
combats.info()

print('\nNulos pokemon:\n', pokemon.isna().sum().sort_values(ascending=False).head(10))
print('\nNulos combats:\n', combats.isna().sum())

print('\nDuplicados pokemon:', pokemon.duplicated().sum())
print('Duplicados combats:', combats.duplicated().sum())

pokemon_ids = set(pokemon['#'].astype(int).tolist())
combat_ids = set(combats['First_pokemon']).union(set(combats['Second_pokemon'])).union(set(combats['Winner']))
missing_ids = sorted(combat_ids - pokemon_ids)

print('\nIDs en combats que no están en pokemon.csv:', len(missing_ids))
if missing_ids:
    print('Primeros IDs faltantes:', missing_ids[:20])

## 3) Construcción del dataset de entrenamiento
Transformamos cada combate a un problema binario:
- `target = 1` si gana `First_pokemon`.
- `target = 0` si gana `Second_pokemon`.

Se generan features por diferencia entre stats de ambos contrincantes para capturar ventaja relativa.

In [ ]:
# Selección y limpieza de columnas base
pokemon_df = pokemon.copy()
pokemon_df = pokemon_df.rename(columns={'#': 'pokemon_id'})
pokemon_df['Type 2'] = pokemon_df['Type 2'].fillna('None')

# Variables numéricas y categóricas de interés
num_stats = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Generation']
cat_stats = ['Type 1', 'Type 2', 'Legendary']

keep_cols = ['pokemon_id', 'Name'] + num_stats + cat_stats
pokemon_df = pokemon_df[keep_cols].copy()
pokemon_df['Legendary'] = pokemon_df['Legendary'].astype(int)

# Merge para primer y segundo pokemon
left = combats.merge(
    pokemon_df.add_prefix('p1_'),
    left_on='First_pokemon',
    right_on='p1_pokemon_id',
    how='left',
)

full = left.merge(
    pokemon_df.add_prefix('p2_'),
    left_on='Second_pokemon',
    right_on='p2_pokemon_id',
    how='left',
)

# Target binario
full['target'] = (full['Winner'] == full['First_pokemon']).astype(int)

# Features por diferencia numérica
for col in num_stats:
    full[f'diff_{col}'] = full[f'p1_{col}'] - full[f'p2_{col}']

# Feature adicional agregada
full['diff_total_stats'] = full[[f'diff_{c}' for c in ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']]].sum(axis=1)

# Features categóricas pareadas
full['p1_type_combo'] = full['p1_Type 1'].astype(str) + '_' + full['p1_Type 2'].astype(str)
full['p2_type_combo'] = full['p2_Type 1'].astype(str) + '_' + full['p2_Type 2'].astype(str)

print('Dataset modelado shape:', full.shape)
display(full.head())

print('Distribución de target (1=gana p1):')
print(full['target'].value_counts(normalize=True).rename('proportion'))

## 4) EDA orientado al objetivo
Analizamos variables derivadas para entender señales predictivas:
- Balance de clases.
- Distribuciones de diferencias de stats.
- Correlación con `target`.

In [ ]:
# Balance de clases
class_dist = full['target'].value_counts(normalize=True).sort_index()
ax = class_dist.plot(kind='bar', color=['#4C72B0', '#DD8452'])
ax.set_xticklabels(['Gana p2 (0)', 'Gana p1 (1)'], rotation=0)
ax.set_ylabel('Proporción')
ax.set_title('Distribución de clases del target')
plt.show()

# Distribuciones de diferencias de stats
diff_cols = [c for c in full.columns if c.startswith('diff_')]
selected_diff = ['diff_HP', 'diff_Attack', 'diff_Defense', 'diff_Sp. Atk', 'diff_Sp. Def', 'diff_Speed', 'diff_total_stats']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(selected_diff):
    sns.kdeplot(data=full, x=col, hue='target', common_norm=False, fill=True, alpha=0.25, ax=axes[i])
    axes[i].set_title(col)
for j in range(len(selected_diff), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

# Correlación simple contra target (features numéricas)
corr_series = full[diff_cols + ['target']].corr(numeric_only=True)['target'].sort_values(ascending=False)
display(corr_series.to_frame('corr_with_target'))

## 5) Baseline y estrategia de modelado
Se entrenan tres modelos con validación cruzada estratificada:
1. **Logistic Regression** (baseline interpretable).
2. **Random Forest** (relaciones no lineales).
3. **HistGradientBoosting** (boosting eficiente).

El criterio principal será `ROC-AUC`, acompañado de `accuracy` y `f1`.

In [ ]:
# Selección de features
num_features = [c for c in full.columns if c.startswith('diff_')]
cat_features = ['p1_Type 1', 'p1_Type 2', 'p2_Type 1', 'p2_Type 2', 'p1_Legendary', 'p2_Legendary', 'p1_Generation', 'p2_Generation']

X = full[num_features + cat_features].copy()
y = full['target'].copy()

# Split holdout final
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Preprocesamiento
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

# One-hot para categóricas en dataframe final
df_train = X_train.copy()
df_test = X_test.copy()

df_train = pd.get_dummies(df_train, columns=['p1_Type 1', 'p1_Type 2', 'p2_Type 1', 'p2_Type 2'], drop_first=False)
df_test = pd.get_dummies(df_test, columns=['p1_Type 1', 'p1_Type 2', 'p2_Type 1', 'p2_Type 2'], drop_first=False)

df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

print('X_train shape (modeling):', df_train.shape)
print('X_test shape (modeling):', df_test.shape)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'}

models = {
    'logreg': LogisticRegression(max_iter=2000, random_state=SEED),
    'random_forest': RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=SEED,
    ),
    'hist_gb': HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=8,
        max_iter=400,
        random_state=SEED,
    ),
}

cv_results = []
fitted_models = {}

for name, model in models.items():
    results = cross_validate(model, df_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results.append({
        'model': name,
        'cv_accuracy_mean': results['test_accuracy'].mean(),
        'cv_accuracy_std': results['test_accuracy'].std(),
        'cv_f1_mean': results['test_f1'].mean(),
        'cv_f1_std': results['test_f1'].std(),
        'cv_roc_auc_mean': results['test_roc_auc'].mean(),
        'cv_roc_auc_std': results['test_roc_auc'].std(),
    })

    model.fit(df_train, y_train)
    fitted_models[name] = model

cv_df = pd.DataFrame(cv_results).sort_values('cv_roc_auc_mean', ascending=False).reset_index(drop=True)
display(cv_df)

best_model_name = cv_df.loc[0, 'model']
best_model = fitted_models[best_model_name]
print(f'Mejor modelo por CV ROC-AUC: {best_model_name}')

In [ ]:
# Evaluación en holdout
proba = best_model.predict_proba(df_test)[:, 1] if hasattr(best_model, 'predict_proba') else None
pred = best_model.predict(df_test)

holdout_accuracy = accuracy_score(y_test, pred)
holdout_f1 = f1_score(y_test, pred)
holdout_auc = roc_auc_score(y_test, proba) if proba is not None else np.nan

print('--- Holdout metrics ---')
print(f'Accuracy: {holdout_accuracy:.4f}')
print(f'F1-score: {holdout_f1:.4f}')
print(f'ROC-AUC: {holdout_auc:.4f}')

print('\n--- Classification report ---')
print(classification_report(y_test, pred, digits=4))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Matriz de confusión - {best_model_name}')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.show()

In [ ]:
# Importancia de variables (permutation importance sobre holdout)
perm = permutation_importance(
    best_model,
    df_test,
    y_test,
    scoring='roc_auc',
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    'feature': df_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

display(importance_df.head(20))

plt.figure(figsize=(10, 6))
plot_df = importance_df.head(15).sort_values('importance_mean', ascending=True)
plt.barh(plot_df['feature'], plot_df['importance_mean'])
plt.title(f'Top 15 importancias por permutación ({best_model_name})')
plt.xlabel('Importancia media en ROC-AUC')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 6) Conclusiones ejecutivas
Completar al finalizar ejecución:

- **Modelo ganador:** `best_model_name`.
- **Métrica principal (ROC-AUC holdout):** `holdout_auc`.
- **Señales más influyentes:** revisar top de permutation importance.
- **Hallazgos de negocio/data:**
  - Diferencias en `Speed`, `Attack` y total de stats suelen capturar gran parte de la ventaja competitiva.
  - Variables de tipo agregan contexto importante en matchups cerrados.

## 7) Próximos pasos (senior roadmap)
1. Incorporar matriz explícita de efectividad de tipos (feature de ventaja real por matchup).
2. Calibrar probabilidades (`CalibratedClassifierCV`) para decisiones por umbral.
3. Hacer validación temporal/por bloque si existe sesgo de muestreo.
4. Versionar experimento (MLflow/W&B) y exportar pipeline productivo.